In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "database" / "CSCAF_public.db"
REPORT_FOLDER = REPO_ROOT / "data_load" / "load_reports"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert DB_PATH.exists(), "run 01_load_ecsf_to_database.ipynb and 02_load_curricula_to_database.ipynb first"

In [ ]:
from cscaf.loading import CurriculumLoader, CurriculumSimilarities
from cscaf.storage import Storage
from cscaf.study import Study
from cscaf.text import TextPipeline

study = Study.from_config()
storage = Storage.of(study, read_only=False)
text = TextPipeline(study)
run_id = storage.runs.ensure_run(study.run_record())
keys = sorted(key for _, key in storage.curricula.curricula() or [])
REPORT_FOLDER.mkdir(parents=True, exist_ok=True)
segmentation = study.section("segmentation")
print("database           :", storage.path.relative_to(REPO_ROOT))
print("segmentation model :", segmentation["model"], "@", segmentation["model_revision"])
print("preprocessing      :", "on" if segmentation.get("preprocessing", {}).get("enabled") else "OFF")
print("line breaks forced :", segmentation["split_on_input_newlines"])
print("embedding model    :", text.model_name, "| stored as", text.embedding_dtype)
print("run                :", run_id)
print("curricula          :", len(keys))

In [ ]:
content = {table: storage.count_rows(table, run_id) for table in
           ("rf_sentence", "curriculum_sentence", "curriculum_rf_similarity_all",
            "curriculum_rf_similarity_top_1", "curriculum_syllabus_rf_top_1")}
content["curriculum"] = storage.count_rows("curriculum")
content

In [ ]:
assert content["rf_sentence"], "no ECSF statements for this run: run 01_load_ecsf_to_database.ipynb first"
segmented = CurriculumLoader(storage, text).segment(run_id, keys)
segmented

In [ ]:
report = CurriculumSimilarities(storage, run_id).compute(REPORT_FOLDER, curriculum_keys=keys)

In [ ]:
statements = storage.count_rows("rf_sentence", run_id)
sentences = {key: (storage.curricula.content_counts(key, run_id) or {}).get("sentences", 0) for key in keys}
check = storage.similarities.check_curriculum_best_matches(run_id)
print("sentences per curriculum:", sentences)
print("top_1:", check)
assert all(sentences.values()), [key for key, n in sentences.items() if not n]
assert storage.count_rows("curriculum_rf_similarity_all", run_id) == statements * storage.count_rows("curriculum_sentence", run_id)
assert check["primary_rows"] == check["pairs_in_all"]
assert check["primary_not_max"] == 0 and check["other_text"] == 0

In [ ]:
for score, key, course, statement, sentence in storage.similarities.best_curriculum_matches(run_id, limit=5):
    print(f"{score:.3f}  {key} / {course}\n       ECSF: {statement}\n       text: {sentence}")